In [1]:
import time
import random

In [2]:
# 1. DATA CONFIGURATION
WEEKDAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
WEEKEND  = ['Sat', 'Sun']
ALL_DAYS = WEEKDAYS + WEEKEND
ALL_SKILLS = ['Barista', 'Cashier', 'Cook', 'Cleaner', 'Supervisor']

In [3]:
# Original 10 employees from the assignment table
EMPLOYEES_BASE = [
    {'name': 'Aleksandra', 'max_hours': 28, 'availability': ['Mon','Tue','Thu','Fri','Sat','Sun'], 'skills': ['Barista','Cashier','Supervisor']},
    {'name': 'Emilio',     'max_hours': 36, 'availability': ['Mon','Wed','Thu','Fri','Sun'],       'skills': ['Barista', 'Cook', 'Cleaner']},
    {'name': 'Leila',      'max_hours': 24, 'availability': ['Sat','Sun'],                         'skills': ['Barista','Cashier','Cleaner']},
    {'name': 'Hancheng',   'max_hours': 36, 'availability': ['Tue','Wed','Thu','Sat','Sun'],       'skills': ['Cashier','Cook','Supervisor']},
    {'name': 'Syed',       'max_hours': 28, 'availability': ['Mon','Tue','Wed','Fri','Sat','Sun'], 'skills': ['Barista','Cashier', 'Cleaner']},
    {'name': 'Zijian',     'max_hours': 16, 'availability': ['Mon','Tue','Wed','Fri','Sat','Sun'], 'skills': ['Cook','Cleaner']},
    {'name': 'Lena',       'max_hours': 20, 'availability': ['Wed','Thu','Fri','Sat','Sun'],       'skills': ['Barista','Supervisor']},
    {'name': 'Marco',      'max_hours': 28, 'availability': ['Tue','Thu','Fri','Sat'],             'skills': ['Cook','Cashier','Cleaner']},
    {'name': 'Clara',      'max_hours': 16, 'availability': ['Sat','Sun'],                         'skills': ['Cook','Cleaner']},
    {'name': 'Laura',      'max_hours': 32, 'availability': ['Sat','Sun'],                         'skills': ['Barista','Cashier','Cleaner']},
]

In [4]:
def generate_extra_employees(total_needed):
    """Generates extra employees to reach the target count for scaling tests."""
    current_list = list(EMPLOYEES_BASE)
    if total_needed <= len(current_list):
        return current_list[:total_needed]
    
    for i in range(len(current_list) + 1, total_needed + 1):
        emp = {
            'name': f'Employee_{i}',
            'max_hours': random.choice([16, 20, 24, 28, 32, 36]),
            'availability': random.sample(ALL_DAYS, random.randint(4, 7)),
            'skills': random.sample(ALL_SKILLS, random.randint(1, 3))
        }
        current_list.append(emp)
    return current_list

In [5]:
# Standard shifts (4 hours each)
STD_WEEKDAY = [('Morning', 4, ['Barista', 'Cashier', 'Cleaner']), ('Afternoon', 4, ['Barista', 'Cook', 'Supervisor'])]
STD_WEEKEND = [('Morning', 4, ['Barista', 'Cashier', 'Supervisor']), ('Afternoon', 4, ['Barista', 'Barista', 'Cook', 'Cleaner']), ('Evening', 4, ['Cashier', 'Cook', 'Supervisor'])]

In [6]:
# Harder Q1: Uneven durations (e.g., 3h and 5h shifts)
UNEVEN_WEEKDAY = [('Morning', 3, ['Barista', 'Cashier', 'Cleaner']), ('Afternoon', 5, ['Barista', 'Cook', 'Supervisor'])]
UNEVEN_WEEKEND = [('Morning', 4, ['Barista', 'Cashier', 'Supervisor']), ('Afternoon', 3, ['Barista', 'Barista', 'Cook', 'Cleaner']), ('Evening', 5, ['Cashier', 'Cook', 'Supervisor'])]

In [7]:
def build_slots(weekday_cfg, weekend_cfg):
    slots = []
    for day in ALL_DAYS:
        shifts = weekend_cfg if day in WEEKEND else weekday_cfg
        for name, hours, roles in shifts:
            for role in roles:
                slots.append({'day': day, 'shift': name, 'role': role, 'hours': hours})
    return slots

In [9]:
# 2. SIMPLE RECURSIVE DFS
def solve_dfs(slots, employees, current_idx=0, schedule=None, hours_used=None):
    if schedule is None: schedule = []
    if hours_used is None: hours_used = {e['name']: 0 for e in employees}

    # BASE CASE: All slots are filled
    if current_idx == len(slots):
        return list(schedule)

    slot = slots[current_idx]

    for emp in employees:
        name = emp['name']
        
        # 1. Check Skill
        if slot['role'] not in emp['skills']: continue
        # 2. Check Availability
        if slot['day'] not in emp['availability']: continue
        # 3. Check Max Hours
        if hours_used[name] + slot['hours'] > emp['max_hours']: continue
        # 4. Check Double-booking
        if any(s['day'] == slot['day'] and s['shift'] == slot['shift'] and s['assigned'] == name for s in schedule):
            continue

        # Valid choice! Update state
        hours_used[name] += slot['hours']
        new_entry = slot.copy()
        new_entry['assigned'] = name
        schedule.append(new_entry)

        # Recurse
        result = solve_dfs(slots, employees, current_idx + 1, schedule, hours_used)
        if result: return result 

        # BACKTRACK
        schedule.pop()
        hours_used[name] -= slot['hours']

    return None

In [10]:
# 3. RUNNING
def run_benchmark(label, weekday_cfg, weekend_cfg):
    print(f"\n--- {label} ---")
    slots = build_slots(weekday_cfg, weekend_cfg)
    
    # Run for 5, 10, and 50 employees as per requirements
    for count in [5, 10, 50]:
        test_emps = generate_extra_employees(count)
        start = time.perf_counter()
        solution = solve_dfs(slots, test_emps)
        end = time.perf_counter()
        
        status = "FOUND" if solution else "NO SOLUTION"
        print(f"{count} Employees: {status} | Time: {end-start:.2f}s")

In [11]:
# Execute Standard Tests
run_benchmark("Standard Configuration", STD_WEEKDAY, STD_WEEKEND)


--- Standard Configuration ---
5 Employees: NO SOLUTION | Time: 0.00s
10 Employees: FOUND | Time: 0.03s
50 Employees: FOUND | Time: 0.00s


In [12]:
# Execute Harder Question 1 Tests (Uneven Slots)
run_benchmark("Harder Q1: Uneven Slots", UNEVEN_WEEKDAY, UNEVEN_WEEKEND)


--- Harder Q1: Uneven Slots ---
5 Employees: NO SOLUTION | Time: 0.00s
10 Employees: FOUND | Time: 19.39s
50 Employees: FOUND | Time: 0.00s
